# RunLens AI — Augmented Analytics para corrida

**Notebook demonstrativo: Google Colab + Strava + Gemini + Knowledge Base local**

Este notebook implementa um MVP acadêmico capaz de:

- receber uma meta de prova e um plano semanal;
- importar corridas do Strava ou gerar dados demonstrativos;
- calcular métricas e produzir gráficos de forma determinística;
- manter uma Knowledge Base local em SQLite FTS5;
- disponibilizar ferramentas para um agente especialista;
- usar a API do Gemini apenas para interpretar evidências já calculadas.

> **Importante:** o agente oferece análise educacional e não substitui treinador, médico ou fisioterapeuta. Ele não deve diagnosticar lesões nem prescrever condutas médicas.

## 1. Preparação

No Colab, abra o painel **Secrets** (ícone de chave) e adicione:

- `GEMINI_API_KEY`: chave criada no Google AI Studio;
- `STRAVA_ACCESS_TOKEN`: opcional, para usar suas atividades reais;
- `GEMINI_MODEL`: opcional; o notebook usa `gemini-3.1-flash-lite` por padrão.

Para uma demonstração, o token de acesso da página de configurações da aplicação Strava é suficiente. Ele expira, então atualize o Secret antes da apresentação. O modo demo não requer Strava.

Não escreva chaves diretamente nas células e limpe os outputs antes de compartilhar o notebook.

> **Privacidade:** segundo a tabela atual de preços do Gemini, dados enviados pelo Free Tier podem ser usados para melhorar produtos. Para a apresentação acadêmica, prefira o modo demonstração ou dados anonimizados e não envie informações sensíveis ao modelo.

In [ ]:
%pip install -q -U google-genai pandas numpy plotly requests pydantic

In [ ]:
import json
import os
import re
import sqlite3
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


def get_secret(name: str, default=None):
    """Lê um Secret do Colab e usa variável de ambiente como fallback."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.getenv(name, default)


GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
STRAVA_ACCESS_TOKEN = get_secret("STRAVA_ACCESS_TOKEN")
GEMINI_MODEL = get_secret("GEMINI_MODEL", "gemini-3.1-flash-lite")

# Mantenha a URL configurável: o Strava anunciou uma mudança futura de domínio.
STRAVA_API_BASE_URL = "https://www.strava.com/api/v3"

print("Gemini:", "configurado" if GEMINI_API_KEY else "sem chave — agente não será chamado")
print("Strava:", "configurado" if STRAVA_ACCESS_TOKEN else "sem token — usando modo demo")
print("Modelo:", GEMINI_MODEL)

## 2. Meta do atleta

Altere os valores abaixo para representar o caso apresentado no trabalho. O ritmo-alvo é calculado por Python e não pelo modelo de linguagem.

In [ ]:
GOAL = {
    "athlete_name": "Atleta Demo",
    "race_name": "Prova-alvo de 10 km",
    "race_distance_km": 10.0,
    "race_date": "2027-03-28",
    "target_time_minutes": 50,
    "available_days_per_week": 4,
    "notes": "Objetivo acadêmico; considerar apenas dados disponíveis.",
}

GOAL["target_pace_min_km"] = round(
    GOAL["target_time_minutes"] / GOAL["race_distance_km"], 2
)

GOAL

## 3. Fonte de dados: Strava ou modo demonstração

`load_strava_activities` pagina as atividades recentes. Para o trabalho, o acesso é somente leitura. O notebook filtra modalidades de corrida e não solicita streams detalhados para evitar chamadas desnecessárias à API.

In [ ]:
RUN_SPORT_TYPES = {"Run", "TrailRun", "VirtualRun"}


def load_strava_activities(access_token: str, lookback_days: int = 90) -> list[dict]:
    """Baixa atividades recentes da conta autenticada no Strava."""
    if not access_token:
        raise ValueError("STRAVA_ACCESS_TOKEN não configurado.")

    headers = {"Authorization": f"Bearer {access_token}"}
    after = int((datetime.now(timezone.utc) - timedelta(days=lookback_days)).timestamp())
    activities = []

    for page in range(1, 11):
        response = requests.get(
            f"{STRAVA_API_BASE_URL}/athlete/activities",
            headers=headers,
            params={"after": after, "page": page, "per_page": 100},
            timeout=30,
        )
        response.raise_for_status()
        batch = response.json()
        activities.extend(batch)
        if len(batch) < 100:
            break

    return [a for a in activities if a.get("sport_type") in RUN_SPORT_TYPES]


def make_demo_activities(weeks: int = 12, seed: int = 42) -> list[dict]:
    """Gera corridas sintéticas e reproduzíveis para apresentação offline."""
    rng = np.random.default_rng(seed)
    current_week = pd.Timestamp.today().normalize() - pd.Timedelta(
        days=pd.Timestamp.today().weekday()
    )
    week_starts = pd.date_range(end=current_week, periods=weeks, freq="7D")
    activities = []
    activity_id = 100_000

    for week_index, week in enumerate(week_starts):
        sessions = 3 if week_index < 4 else 4
        progressive_base = 5.0 + 0.13 * week_index
        offsets = [0, 2, 4, 6][:sessions]

        for session_index, offset in enumerate(offsets):
            if session_index == sessions - 1:
                distance_km = 8.0 + 0.55 * week_index + rng.normal(0, 0.5)
                workout_name = "Longo"
                pace = 6.25 - 0.015 * week_index + rng.normal(0, 0.08)
            elif session_index == 1 and week_index >= 3:
                distance_km = progressive_base + rng.normal(0, 0.35)
                workout_name = "Ritmo/intervalado"
                pace = 5.35 - 0.025 * week_index + rng.normal(0, 0.08)
            else:
                distance_km = progressive_base + rng.normal(0, 0.4)
                workout_name = "Corrida leve"
                pace = 6.05 - 0.012 * week_index + rng.normal(0, 0.1)

            distance_km = max(distance_km, 3.0)
            moving_time_sec = distance_km * pace * 60
            average_speed = (distance_km * 1000) / moving_time_sec
            start = week + pd.Timedelta(days=offset, hours=7)

            activities.append(
                {
                    "id": activity_id,
                    "name": workout_name,
                    "sport_type": "Run",
                    "start_date_local": start.isoformat(),
                    "distance": float(distance_km * 1000),
                    "moving_time": int(moving_time_sec),
                    "elapsed_time": int(moving_time_sec + rng.integers(0, 240)),
                    "total_elevation_gain": float(max(0, rng.normal(55, 20))),
                    "average_speed": float(average_speed),
                    "average_heartrate": float(148 + (6.0 - pace) * 8 + rng.normal(0, 3)),
                }
            )
            activity_id += 1

    return activities


if STRAVA_ACCESS_TOKEN:
    try:
        raw_activities = load_strava_activities(STRAVA_ACCESS_TOKEN, lookback_days=90)
        DATA_SOURCE = "Strava"
    except Exception as exc:
        print(f"Falha ao consultar Strava: {exc}")
        print("Continuando com dados demonstrativos.")
        raw_activities = make_demo_activities()
        DATA_SOURCE = "Demonstração"
else:
    raw_activities = make_demo_activities()
    DATA_SOURCE = "Demonstração"

print(f"Fonte: {DATA_SOURCE} | atividades carregadas: {len(raw_activities)}")

## 4. Normalização e qualidade dos dados

A camada abaixo converte o JSON do Strava em uma tabela analítica. Valores ausentes permanecem ausentes: o sistema não inventa frequência cardíaca, elevação ou ritmo.

In [ ]:
def normalize_activities(activities: list[dict]) -> pd.DataFrame:
    columns = [
        "id", "name", "sport_type", "start_date_local", "distance",
        "moving_time", "elapsed_time", "total_elevation_gain",
        "average_speed", "average_heartrate",
    ]
    frame = pd.DataFrame(activities)
    for column in columns:
        if column not in frame.columns:
            frame[column] = np.nan

    frame = frame[columns].copy()
    frame["date"] = pd.to_datetime(
        frame["start_date_local"], errors="coerce", utc=True
    ).dt.tz_convert(None)
    frame["distance_km"] = pd.to_numeric(frame["distance"], errors="coerce") / 1000
    frame["moving_time_sec"] = pd.to_numeric(frame["moving_time"], errors="coerce")
    frame["duration_min"] = frame["moving_time_sec"] / 60
    frame["elevation_m"] = pd.to_numeric(
        frame["total_elevation_gain"], errors="coerce"
    )
    frame["average_hr"] = pd.to_numeric(
        frame["average_heartrate"], errors="coerce"
    )
    frame["pace_min_km"] = np.where(
        frame["distance_km"] > 0,
        frame["duration_min"] / frame["distance_km"],
        np.nan,
    )
    frame["week"] = frame["date"].dt.normalize() - pd.to_timedelta(
        frame["date"].dt.weekday, unit="D"
    )
    return frame.sort_values("date").reset_index(drop=True)


runs_df = normalize_activities(raw_activities)

quality_summary = {
    "source": DATA_SOURCE,
    "runs": int(len(runs_df)),
    "period_start": runs_df["date"].min().date().isoformat() if len(runs_df) else None,
    "period_end": runs_df["date"].max().date().isoformat() if len(runs_df) else None,
    "heart_rate_coverage_pct": float(round(100 * runs_df["average_hr"].notna().mean(), 1)) if len(runs_df) else 0.0,
    "valid_pace_coverage_pct": float(round(100 * runs_df["pace_min_km"].notna().mean(), 1)) if len(runs_df) else 0.0,
}

display(pd.DataFrame([quality_summary]))
display(runs_df.tail(8)[["date", "name", "distance_km", "duration_min", "pace_min_km", "average_hr"]])

## 5. Plano semanal e aderência

Para o MVP, o plano é uma tabela semanal. Em uma evolução futura, o usuário poderá enviar um CSV com cada sessão planejada. Ajuste `planned_km` e `planned_sessions` conforme o plano real.

In [ ]:
observed_weeks = pd.Series(runs_df["week"].dropna().sort_values().unique())
plan_df = pd.DataFrame({"week": pd.to_datetime(observed_weeks)})
plan_df["planned_km"] = np.linspace(22, 36, len(plan_df)).round(1)
plan_df["planned_sessions"] = np.where(np.arange(len(plan_df)) < 4, 3, 4)

# Exemplo de semana de recuperação, para tornar a comparação mais realista.
if len(plan_df) >= 8:
    plan_df.loc[7, "planned_km"] = round(plan_df.loc[6, "planned_km"] * 0.82, 1)

display(plan_df.tail(8))

In [ ]:
def calculate_weekly_metrics(runs: pd.DataFrame) -> pd.DataFrame:
    if runs.empty:
        return pd.DataFrame()

    weekly = (
        runs.groupby("week", as_index=False)
        .agg(
            actual_km=("distance_km", "sum"),
            actual_sessions=("id", "count"),
            duration_min=("duration_min", "sum"),
            elevation_m=("elevation_m", "sum"),
            longest_run_km=("distance_km", "max"),
        )
        .sort_values("week")
    )
    weekly["pace_min_km"] = weekly["duration_min"] / weekly["actual_km"]
    weekly["volume_change_pct"] = weekly["actual_km"].pct_change() * 100
    return weekly


def compare_plan_to_actual(plan: pd.DataFrame, weekly: pd.DataFrame) -> pd.DataFrame:
    comparison = plan.merge(weekly, on="week", how="left")
    numeric_columns = [
        "actual_km", "actual_sessions", "duration_min", "elevation_m",
        "longest_run_km",
    ]
    comparison[numeric_columns] = comparison[numeric_columns].fillna(0)
    comparison["distance_adherence_pct"] = np.where(
        comparison["planned_km"] > 0,
        100 * comparison["actual_km"] / comparison["planned_km"],
        np.nan,
    )
    comparison["session_gap"] = (
        comparison["actual_sessions"] - comparison["planned_sessions"]
    )
    comparison["alignment_label"] = pd.cut(
        comparison["distance_adherence_pct"],
        bins=[-np.inf, 80, 120, np.inf],
        labels=["abaixo do plano", "próximo do plano", "acima do plano"],
    ).astype(str)
    return comparison


weekly_df = calculate_weekly_metrics(runs_df)
comparison_df = compare_plan_to_actual(plan_df, weekly_df)

display(comparison_df.tail(8))

## 6. Visualizações determinísticas

Os gráficos são produzidos por funções Python. O Gemini interpreta os resultados, mas não executa código arbitrário nem inventa séries para plotagem.

In [ ]:
def chart_plan_vs_actual(comparison: pd.DataFrame):
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=comparison["week"],
            y=comparison["actual_km"],
            name="Realizado (km)",
            marker_color="#FF5A36",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=comparison["week"],
            y=comparison["planned_km"],
            name="Planejado (km)",
            mode="lines+markers",
            line={"color": "#172A3A", "width": 3},
        )
    )
    fig.update_layout(
        title="Volume semanal: planejado x realizado",
        xaxis_title="Semana",
        yaxis_title="Distância (km)",
        barmode="group",
        template="plotly_white",
        hovermode="x unified",
    )
    return fig


def chart_pace_trend(runs: pd.DataFrame):
    chart_data = runs.dropna(subset=["pace_min_km"]).copy()
    fig = px.scatter(
        chart_data,
        x="date",
        y="pace_min_km",
        size="distance_km",
        color="name",
        hover_data=["distance_km", "average_hr"],
        title="Ritmo das atividades ao longo do tempo",
        labels={"pace_min_km": "Ritmo (min/km)", "date": "Data"},
        template="plotly_white",
    )
    fig.update_yaxes(autorange="reversed")
    return fig


chart_plan_vs_actual(comparison_df).show()
chart_pace_trend(runs_df).show()

## 7. Knowledge Base local com SQLite FTS5

Esta versão não usa Supabase nem banco vetorial. O SQLite FTS5 cria um índice textual local com ranqueamento. Os textos abaixo são demonstrativos; antes da entrega final, substitua ou amplie o conteúdo com a bibliografia aprovada pelo professor.

In [ ]:
KNOWLEDGE_DOCUMENTS = [
    {
        "title": "Separar evidência, inferência e recomendação",
        "source": "Material metodológico RunLens — demonstração",
        "content": (
            "Uma análise deve começar pelos valores observados. Inferências devem ser "
            "apresentadas como hipóteses e recomendações devem explicar quais evidências "
            "foram consideradas. Dados ausentes não podem ser tratados como valores normais."
        ),
    },
    {
        "title": "Aderência ao plano",
        "source": "Material metodológico RunLens — demonstração",
        "content": (
            "Aderência pode ser analisada comparando distância, número de sessões e tipo "
            "de treino planejado com o realizado. Uma única semana não deve determinar uma "
            "conclusão; tendências de várias semanas são mais informativas."
        ),
    },
    {
        "title": "Contexto para interpretar ritmo",
        "source": "Material metodológico RunLens — demonstração",
        "content": (
            "Ritmo isolado não representa todo o esforço. Terreno, elevação, clima, duração, "
            "frequência cardíaca e objetivo da sessão alteram a interpretação. Quando esses "
            "dados não existem, o agente deve declarar a limitação."
        ),
    },
    {
        "title": "Limites de segurança",
        "source": "Política de segurança RunLens — demonstração",
        "content": (
            "O sistema não diagnostica dor, lesão, doença ou condição médica. Relatos de dor, "
            "tontura, mal-estar ou piora persistente devem ser direcionados a um profissional "
            "qualificado. A análise é educacional e baseada apenas nos dados fornecidos."
        ),
    },
]


kb_connection = sqlite3.connect(":memory:", check_same_thread=False)
kb_connection.execute(
    """
    CREATE VIRTUAL TABLE IF NOT EXISTS knowledge USING fts5(
        title,
        content,
        source,
        tokenize='unicode61 remove_diacritics 2'
    )
    """
)
kb_connection.execute("DELETE FROM knowledge")
kb_connection.executemany(
    "INSERT INTO knowledge(title, content, source) VALUES (?, ?, ?)",
    [(d["title"], d["content"], d["source"]) for d in KNOWLEDGE_DOCUMENTS],
)
kb_connection.commit()


def search_knowledge(query: str, top_k: int = 3) -> list[dict]:
    terms = re.findall(r"[\wÀ-ÿ]+", query.lower())
    if not terms:
        return []
    fts_query = " OR ".join(f'"{term}"' for term in terms[:12])
    rows = kb_connection.execute(
        """
        SELECT title, content, source, bm25(knowledge) AS score
        FROM knowledge
        WHERE knowledge MATCH ?
        ORDER BY score
        LIMIT ?
        """,
        (fts_query, int(top_k)),
    ).fetchall()
    return [
        {"title": title, "content": content, "source": source}
        for title, content, source, _ in rows
    ]


search_knowledge("Como interpretar ritmo e frequência cardíaca?")

## 8. Ferramentas do agente

Estas funções são a fronteira entre o Gemini e os dados. Elas retornam apenas JSON simples, permitindo auditar exatamente quais evidências foram disponibilizadas ao modelo.

In [ ]:
def dataframe_records(frame: pd.DataFrame) -> list[dict]:
    """Converte DataFrame em registros JSON sem tipos NumPy/Timestamp."""
    return json.loads(frame.to_json(orient="records", date_format="iso"))


def get_athlete_goal() -> dict:
    """Retorna a prova-alvo, tempo-alvo, ritmo-alvo e disponibilidade do atleta."""
    return GOAL.copy()


def get_data_quality() -> dict:
    """Retorna cobertura, período e origem dos dados antes de qualquer conclusão."""
    return quality_summary.copy()


def get_weekly_training_summary(limit_weeks: int = 8) -> list[dict]:
    """Retorna métricas semanais calculadas. Use para analisar tendência de volume e consistência."""
    limit = max(1, min(int(limit_weeks), 16))
    return dataframe_records(weekly_df.tail(limit))


def compare_training_plan(limit_weeks: int = 8) -> list[dict]:
    """Compara distância e sessões planejadas com o realizado nas semanas recentes."""
    limit = max(1, min(int(limit_weeks), 16))
    columns = [
        "week", "planned_km", "actual_km", "planned_sessions",
        "actual_sessions", "distance_adherence_pct", "session_gap",
        "alignment_label", "longest_run_km",
    ]
    return dataframe_records(comparison_df.tail(limit)[columns])


def get_recent_runs(limit: int = 5) -> list[dict]:
    """Retorna as corridas mais recentes com distância, duração, ritmo e FC quando disponível."""
    limit = max(1, min(int(limit), 20))
    columns = [
        "date", "name", "distance_km", "duration_min", "pace_min_km",
        "average_hr", "elevation_m",
    ]
    return dataframe_records(runs_df.tail(limit)[columns])


def search_training_knowledge(query: str, top_k: int = 3) -> list[dict]:
    """Busca explicações e limites metodológicos na Knowledge Base local."""
    return search_knowledge(query=query, top_k=top_k)


TOOL_FUNCTIONS = [
    get_athlete_goal,
    get_data_quality,
    get_weekly_training_summary,
    compare_training_plan,
    get_recent_runs,
    search_training_knowledge,
]

print("Ferramentas disponíveis:", [tool.__name__ for tool in TOOL_FUNCTIONS])

## 9. Agente especialista com Gemini

O SDK recebe funções Python como ferramentas. O Gemini escolhe as funções necessárias; o SDK executa as funções localmente e devolve os resultados ao modelo. O prompt exige rastreabilidade e impede que dados ausentes sejam inventados.

In [ ]:
from google import genai
from google.genai import types


SYSTEM_PROMPT = """
Você é o RunLens Coach, um especialista em análise de dados de corrida.

OBJETIVO
Transformar o plano, a meta e o histórico do atleta em insights claros, auditáveis
e educacionais. Você não é médico e não substitui acompanhamento profissional.

REGRAS OBRIGATÓRIAS
1. Use ferramentas antes de fazer afirmações sobre o atleta.
2. Comece verificando a qualidade e a cobertura dos dados.
3. Diferencie explicitamente: Evidência observada, Interpretação e Próximo passo.
4. Cite datas, semanas e métricas que sustentam cada conclusão.
5. Nunca invente frequência cardíaca, tipo de treino, percepção de esforço ou lesão.
6. Se o dado necessário estiver ausente, declare a limitação.
7. Use a Knowledge Base ao explicar metodologia ou limites da análise e cite a fonte.
8. Não diagnostique nem forneça tratamento médico.
9. Não trate ritmo-alvo como ritmo adequado para todas as sessões.
10. Responda em português do Brasil, de forma concisa e acolhedora.

FORMATO PREFERENCIAL
- Resumo
- Evidências
- Interpretação
- Próximo passo sugerido
- Limitações e fontes
""".strip()


gemini_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None


def ask_runlens(question: str) -> str:
    """Faz uma pergunta ao agente com function calling automático."""
    if not gemini_client:
        return (
            "GEMINI_API_KEY não configurada. Adicione a chave no painel Secrets "
            "do Colab e execute novamente as células de configuração e do agente."
        )

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            tools=TOOL_FUNCTIONS,
            temperature=0.2,
        ),
    )
    return response.text


print("Agente pronto." if gemini_client else "Agente aguardando GEMINI_API_KEY.")

## 10. Faça uma pergunta

A célula abaixo consome uma chamada da API apenas quando executada. Exemplos:

- `Estou aderindo ao meu plano nas últimas 6 semanas?`
- `Quais tendências aparecem no meu volume e no treino longo?`
- `Os dados disponíveis sustentam dizer que estou pronto para o tempo-alvo?`
- `Quais são as limitações desta análise?`

In [ ]:
QUESTION = "Estou aderindo ao plano nas últimas 6 semanas? Mostre as evidências."

answer = ask_runlens(QUESTION)
print(answer)

## 11. Validação para a apresentação

Antes de entregar, execute este checklist:

1. O agente informa a fonte e a cobertura dos dados?
2. Todas as métricas mencionadas existem nas tabelas?
3. Sem frequência cardíaca, ele declara a limitação?
4. A comparação usa o plano real, e não um plano inventado?
5. As recomendações estão separadas das evidências?
6. A resposta cita documentos recuperados da Knowledge Base?
7. O modo demonstração funciona sem Strava?
8. Nenhuma chave aparece no notebook ou nos outputs?

### Próximas evoluções

- upload do plano por CSV;
- detalhamento por atividade usando streams do Strava;
- interface Streamlit com URL estável para OAuth;
- persistência em SQLite no Google Drive;
- avaliação automática de factualidade das respostas;
- substituição dos documentos demonstrativos por bibliografia técnica aprovada.

## Referências técnicas

- [Gemini API — início e function calling](https://ai.google.dev/gemini-api/docs/get-started)
- [Gemini API — preços e Free Tier](https://ai.google.dev/gemini-api/docs/pricing)
- [Strava API — autenticação](https://developers.strava.com/docs/authentication/)
- [Strava API — endpoints de atividades](https://developers.strava.com/docs/reference/)
- [SQLite FTS5](https://www.sqlite.org/fts5.html)
- [Streamlit Community Cloud](https://docs.streamlit.io/deploy/streamlit-community-cloud)